In [ ]:
# Install dependencies
%pip install anthropic python-dotenv
%pip install dotenv

In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API Client
from anthropic import Anthropic

client=Anthropic()
model="claude-haiku-4-5"


In [4]:
# make a request
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]   
)

message.content[0].text 

'Quantum computing is a type of computing that uses quantum bits (qubits) that can exist in multiple states simultaneously, allowing quantum computers to process certain types of problems exponentially faster than classical computers.'

In [11]:
# Define helper functions to support multi-message converstation
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return message.content[0].text

In [ ]:
# Make a starting list of messages
messages = []

# Add the initial user message
add_user_message(messages, "Define quantum computing in one sentence.")

# Pass the list of messates into 'chat' to get an answer
answer = chat(messages)

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# Add in user's follow-up question
add_user_message(messages, "Write another sentence.")

# Pass the list of messates into 'chat' to get an answer
answer = chat(messages)

# Add in the answer to the follow up question
add_assistant_message(messages, answer)

messages


In [ ]:
# Chat Exercise
messages = []

while True:
    # Get user input
    user_input = input(">")
    print(">", user_input)

    # add user_messages to messages list
    add_user_message(messages, user_input)

    # call chat to get answer
    answer = chat(messages)

    # add answer to messages list
    add_assistant_message(messages, answer)

    # print answer
    print("---")
    print(answer)
    print("---")



In [ ]:
# Math Tutor Exercise (chat with System Prompts)
system_prompt = "You are a patient math tutor. Do not answer the question directly, but guilde the them to a solution step by step."

def chat_v2(messages, prompt=None):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
    }

    if prompt:
        params["system"] = prompt

    message = client.messages.create(**params)
    return message.content[0].text    
    
messages = []
add_user_message(messages,"How do I create an integral expression from a limit sum expression?")
answer = chat_v2(messages, system_prompt) # test with and without providing the system prompt
add_assistant_message(messages,answer)
print(answer)


In [ ]:
# system prompt exercise 2 - concise code example

messages = []

add_user_message(messages, "Write a Python function that checks a string for duplicate characters")
answer = chat_v2(messages, prompt="You are a Python engineer who rites very concise code")
add_assistant_message(messages, answer)
print(answer)

In [40]:
# Temperature exercise

def chat_v3(messages, prompt=None, temperature=1.0):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature": temperature
    }

    if prompt:
        params["system"] = prompt

    message = client.messages.create(**params)
    return message.content[0].text    

messages = []

add_user_message(messages, "Generate a one sentence movie idea")
answer = chat_v3(messages, temperature=1.0) # test with different temperatures between 0.0 and 1.0
add_assistant_message(messages, answer)
print(answer)

# Movie Idea

A cynical food critic discovers that the mysterious restaurant she's been investigating is run by her estranged mother, who's been using ancient recipes to literally cook away people's deepest regrets.


In [ ]:
# Raw Steaming Respoonse Exercise

message = []

add_user_message(messages, "Given me an example of a hidden variable theory related to the unification of quantum and relativistic physics")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)

for event in stream:
    print(event)




RawMessageStartEvent(message=Message(id='msg_01V6tzpL6CfmbCEYsBHQPBwa', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=189, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Hidden Variable Theory Example: Pilot-Wave Theory (de Broglie-Bohm', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Mechanics)\n\n**De Broglie-Bohm mechani

In [53]:
# Managed Steaming Respoonse Exercise

message = []

add_user_message(messages, "Given me an example of a hidden variable theory related to the unification of quantum and relativistic physics")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="") # outputs chuncks of reponse as they are received

stream.get_final_message() # collects entire final collection of messages so it can stored or consumed seperately

# Hidden Variable Theory Example: De Broglie-Bohm Pilot Wave Theory

**De Broglie-Bohm mechanics** (or pilot wave theory) is a hidden variable approach that attempts to bridge quantum and relativistic physics:

## Key Concept
Particles follow deterministic trajectories guided by a "pilot wave" (the quantum wave function). The wave function is a real physical field that exists in spacetime, not just a mathematical tool for calculating probabilities.

## How It Addresses Unification
- **Hidden Variables**: Particle positions and velocities exist objectively (hidden from standard quantum mechanics)
- **Determinism**: Replaces probabilistic quantum mechanics with deterministic dynamics
- **Spacetime**: The wave function exists in configuration space but can be formulated to respect relativistic principles
- **Relativistic Extensions**: Versions like the Dirac-Bohm theory incorporate relativistic quantum field theory

## Advantage for Unification
By treating the wave function as a real phys

ParsedMessage(id='msg_01AoRXZedyY9kGgAQfm2rUBK', container=None, content=[ParsedTextBlock(citations=None, text='# Hidden Variable Theory Example: De Broglie-Bohm Pilot Wave Theory\n\n**De Broglie-Bohm mechanics** (or pilot wave theory) is a hidden variable approach that attempts to bridge quantum and relativistic physics:\n\n## Key Concept\nParticles follow deterministic trajectories guided by a "pilot wave" (the quantum wave function). The wave function is a real physical field that exists in spacetime, not just a mathematical tool for calculating probabilities.\n\n## How It Addresses Unification\n- **Hidden Variables**: Particle positions and velocities exist objectively (hidden from standard quantum mechanics)\n- **Determinism**: Replaces probabilistic quantum mechanics with deterministic dynamics\n- **Spacetime**: The wave function exists in configuration space but can be formulated to respect relativistic principles\n- **Relativistic Extensions**: Versions like the Dirac-Bohm theo